In [7]:
import json
import os
from pathlib import Path
import urllib.error
import urllib.request

import openai


def load_env_file(path: str = ".env"):
    env_path = Path(path)
    if not env_path.exists():
        return

    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file()

MOVIE_API_BASE_URL = os.getenv(
    "MOVIE_API_BASE_URL", "https://nomad-movies.nomadcoders.workers.dev"
)

client = openai.OpenAI()
messages = []
_tool_turn_buffer: list = []
excluded_movie_ids: set[str] = set()

In [8]:
def _fetch_movie_api(path: str):
    url = f"{MOVIE_API_BASE_URL}{path}"
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/json",
            "User-Agent": "Mozilla/5.0",
        },
    )
    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            return json.loads(response.read().decode("utf-8"))
    except urllib.error.HTTPError as e:
        e.read()
        raise RuntimeError(
            f"Movie API returned HTTP {e.code} for {path}. "
            "Use only numeric TMDB ids from get_popular_movies or get_movie_details "
            "(do not guess ids or use titles)."
        ) from e


def get_popular_movies():
    movies = _fetch_movie_api("/movies")
    filtered = [m for m in movies if str(m.get("id")) not in excluded_movie_ids]
    return json.dumps(filtered)


def register_excluded_movies(movie_ids: list):
    for mid in movie_ids:
        excluded_movie_ids.add(str(mid))
    return json.dumps({"ok": True, "total_excluded": len(excluded_movie_ids)})


def get_movie_details(id):
    details = _fetch_movie_api(f"/movies/{id}")
    return json.dumps(details)


def get_movie_credits(id):
    credits = _fetch_movie_api(f"/movies/{id}/credits")
    return json.dumps(credits)


def get_similar_movies(id):
    similar = _fetch_movie_api(f"/movies/{id}/similar")
    return json.dumps(similar)


FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_similar_movies": get_similar_movies,
    "register_excluded_movies": register_excluded_movies,
}

In [9]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies", # function: get_popular_movies()
            "description": "Get the most popular movie.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": [],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details", # function: get_movie_details(id)
            "description": "Get details for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits", # function: credits(id)
            "description": "Get credit score for a movie by its id.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The movie id.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "get_similar_movies", # function: get_similar_movies(id)
            "description": (
                "Get movies similar to a film (same genres/themes). "
                "The id must be the numeric TMDB id from a prior get_popular_movies or "
                "get_movie_details response — calling this with a wrong or made-up id fails."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "Numeric TMDB movie id from another tool response.",
                    }
                },
                "required": ["id"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "register_excluded_movies",
            "description": (
                "Record TMDB movie ids that must not be suggested again: any title you recommended "
                "in this chat, and any film the user says they already watched. Call with ids from "
                "API responses (same turn as recommendations when possible)."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "movie_ids": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "TMDB movie ids as strings.",
                    }
                },
                "required": ["movie_ids"],
            },
        },
    },
]

In [10]:
from openai.types.chat import ChatCompletionMessage

BASE_SYSTEM = (
    "You are a helpful movie assistant. "
    "Whenever you recommend movies, call register_excluded_movies with each title's TMDB id "
    "from the API data (in the same assistant turn as the recommendation). "
    "When the user says they already watched a movie, look up its id from earlier tool data or "
    "fetch details, then call register_excluded_movies. "
    "Do not recommend movies whose ids appear in the exclusion list below.\n\n"
)


def messages_for_chat_completion():
    if excluded_movie_ids:
        try:
            ordered = sorted(excluded_movie_ids, key=int)
        except ValueError:
            ordered = sorted(excluded_movie_ids)
        suffix = (
            "Excluded TMDB ids (already recommended or user already saw): "
            + ", ".join(ordered)
            + "."
        )
    else:
        suffix = "No movies excluded yet."
    return [{"role": "system", "content": BASE_SYSTEM + suffix}] + messages + _tool_turn_buffer


def process_ai_response(message: ChatCompletionMessage):

    if message.tool_calls:
        _tool_turn_buffer.append(
            {
                "role": "assistant",
                "content": message.content or "",
                "tool_calls": [
                    {
                        "id": tool_call.id,
                        "type": "function",
                        "function": {
                            "name": tool_call.function.name,
                            "arguments": tool_call.function.arguments,
                        },
                    }
                    for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            # print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            if function_to_run is None:
                result = f"Error: unknown function '{function_name}'."
                print(result)
            else:
                try:
                    result = function_to_run(**arguments)
                except Exception as e:
                    result = f"Error running {function_name}: {e}"
                    print(result)

            # print(f"Ran {function_name} with args {arguments} for a result of {result}")

            _tool_turn_buffer.append(
                {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": str(result),
                }
            )
        call_ai()
    else:
        messages.append({"role": "assistant", "content": message.content})
        _tool_turn_buffer.clear()
        print(f"AI: {message.content}")


def call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages_for_chat_completion(),
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [11]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role": "user", "content": message})
        _tool_turn_buffer.clear()
        print(f"User: {message}")
        call_ai()

User: Recommend me some movies that released after March 2026.
AI: I can't provide movie recommendations released after March 2026 at the moment, as my data only includes films released up until October 2023. If you're looking for recent films up to that point or have specific genres in mind, let me know, and I can help with that!
User: what are some movies that are popular nowadays?
AI: Here are some popular movies you might enjoy:

1. **Your Heart Will Be Broken** (2026-03-26)
   ![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)
   - *Overview:* A high school student makes a deal with the bully to pretend to be her boyfriend, leading to unexpected feelings.
   - *Rating:* 6.9

2. **Hoppers** (2026-03-04)
   ![Hoppers](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)
   - *Overview:* A young girl uses technology to communicate with animals, uncovering mysteries beyond her wildest dreams.
   - *Rating:* 7.57

3. **The Super Ma

In [12]:
messages

[{'role': 'user',
  'content': 'Recommend me some movies that released after March 2026.'},
 {'role': 'assistant',
  'content': "I can't provide movie recommendations released after March 2026 at the moment, as my data only includes films released up until October 2023. If you're looking for recent films up to that point or have specific genres in mind, let me know, and I can help with that!"},
 {'role': 'user',
  'content': 'what are some movies that are popular nowadays?'},
 {'role': 'assistant',
  'content': 'Here are some popular movies you might enjoy:\n\n1. **Your Heart Will Be Broken** (2026-03-26)\n   ![Your Heart Will Be Broken](https://image.tmdb.org/t/p/w780/7wIBfBl2gejt6xHxNSK0reVIm7E.jpg)\n   - *Overview:* A high school student makes a deal with the bully to pretend to be her boyfriend, leading to unexpected feelings.\n   - *Rating:* 6.9\n\n2. **Hoppers** (2026-03-04)\n   ![Hoppers](https://image.tmdb.org/t/p/w780/xjtWQ2CL1mpmMNwuU5HeS4Iuwuu.jpg)\n   - *Overview:* A young 